## Session 9, in class: perturbation, Blanchard and Kahn, and Klein

Computational Macro (HWS 2026), University of Mannheim. Thursday, 05.11.2026, 10:15 to 11:45.

This is the code behind the session 9 slides. Part 1 writes the stochastic growth model as one system of
equations and finds its steady state. Part 2 linearizes the system by finite differences. Part 3 solves the
linear system with the eigen-decomposition of Blanchard and Kahn, part 4 with the generalized Schur
decomposition of Klein, including a version of the model that only Klein can handle. Part 5 uses the solution:
impulse responses, a simulation against the exact path of the economy with the closed form, and the economy
with $\gamma = 2$ and $\delta = 0.1$, compared with a global solution.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **10 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: they are part of the method, and the comments
  explain each step.
- Each part ends with a **Check**: a number to compare with, from the slides. Do not move on before it fits.
- **Slides.** "Slide 19" and the like refer to the session 9 slides, by the number at the bottom of each slide.
  Keep them open next to the notebook.
- The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | The model as one system $\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t) = \mathbf{0}$, and its steady state | 1, 2 | 4 to 8 | 10 min |
| 2 | Linearization by finite differences: the matrices $\mathbf{A}$ and $\mathbf{B}$ | 3, 4 | 8 to 10 | 10 min |
| 3 | Blanchard and Kahn: eigenvalues, the count, $\mathbf{G}$ and $\mathbf{H}$ | 5, 6 | 14, 15 | 20 min |
| 4 | Klein: the generalized Schur decomposition, and a model with static equations | 7, 8 | 17 to 21 | 20 min |
| 5 | Impulse responses, simulation against the exact path, the economy with $\gamma = 2$, $\delta = 0.1$ | 9, 10 | 22 to 25 | 25 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\mathbf{x}_t = (k_t, z_t, c_t)$ | the variables of the model, states first | `x`, a vector of logs |
| $\hat{\mathbf{x}}_t = \ln \mathbf{x}_t - \ln \bar{\mathbf{x}}$ | log deviation from the steady state | `x - xss` |
| $\mathbf{s}_t = (k_t, z_t)$ | the states, $n_s = 2$ | `s`, `n_states` |
| $\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t)$ | the equilibrium conditions, zero along an equilibrium path | `F(x_next, x, par)` |
| $\mathbf{A}$, $\mathbf{B}$ | $\mathbf{A} \, \mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \mathbf{B} \, \hat{\mathbf{x}}_t$ | `A`, `B` |
| $\boldsymbol{\Gamma} = \mathbf{A}^{-1} \mathbf{B}$, $\lambda_i$ | the matrix of the difference equation and its eigenvalues | `Gamma`, `lam` |
| $\mathbf{S}$, $\mathbf{T}$, $\mathbf{Z}$ | the generalized Schur decomposition of $(\mathbf{A}, \mathbf{B})$ | `S`, `T`, `Z` |
| $\mathbf{G}$, $\mathbf{H}$ | $\hat{c}_t = \mathbf{G} \hat{\mathbf{s}}_t$ and $\hat{\mathbf{s}}_{t+1} = \mathbf{H} \hat{\mathbf{s}}_t$ | `G`, `H` |

**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `np.exp(x)`, `np.log(x)` | elementwise on a vector: from logs to levels and back | the model in logs |
| `eig(Gamma)` from `scipy.linalg` | eigenvalues and eigenvectors (columns) of a square matrix | Blanchard and Kahn |
| `np.argsort(v)` | the order that sorts `v` | stable eigenvalues first |
| `inv(V)`, `solve(A, B)` | the inverse, and $\mathbf{A}^{-1} \mathbf{B}$ without forming the inverse | the formulas of slides 15 and 20 |
| `ordqz(A, B, sort=f, output="complex")` | the generalized Schur decomposition with the eigenvalues that pass `f` first | Klein |
| `np.linalg.matrix_rank(A)` | the rank of a matrix | a singular $\mathbf{A}$ |
| `np.random.default_rng(seed).standard_normal(T)` | `T` draws from $N(0, 1)$, reproducible through the seed | the simulation |

Everything else is from the earlier notebooks: `A @ B`, `x[:, None]`, `np.interp`, `@dataclass`.

### 0. Packages

Nothing new to install: `scipy.linalg` ships with `scipy`, which is in the course environment since week 3.
The first cell installs the packages into the environment of the running kernel if they are missing.

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "scipy==1.15.*" "matplotlib==3.10.*"

In [ ]:
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eig, inv, solve, ordqz

np.set_printoptions(precision=4, suppress=True)           # four decimals, no scientific notation, in every print

### 1. The model as one system of equations (gaps 1 and 2)

*Slides 4 to 8.*

The stochastic growth model: a household with CRRA utility $u(c) = c^{1 - \gamma} / (1 - \gamma)$ owns the
capital stock, output is $z_t k_t^\alpha$, capital depreciates at rate $\delta$, and log TFP follows
$\ln z_{t+1} = \rho \ln z_t + \varepsilon_{t+1}$ with $\varepsilon_{t+1} \sim N(0, \sigma^2)$. The three
equilibrium conditions, with $R_{t+1} = \alpha z_{t+1} k_{t+1}^{\alpha - 1} + 1 - \delta$:

$$
\begin{aligned}
k_{t+1} &= z_t k_t^\alpha + (1 - \delta) k_t - c_t && \text{resource constraint} \\
\ln z_{t+1} &= \rho \ln z_t + \varepsilon_{t+1} && \text{TFP} \\
u_c(c_t) &= \beta \, \mathbb{E}_t \bigl[ u_c(c_{t+1}) \, R_{t+1} \bigr] && \text{Euler equation}
\end{aligned}
$$

The parameters are $\alpha = 0.3$ and $\beta = 0.96$ as in sessions 1 and 2, $\rho = 0.9$, $\sigma = 0.02$.
Parts 1 to 4 and the first half of part 5 use $\gamma = 1$ and $\delta = 1$: log utility and full
depreciation. Then the policy is known in closed form, $k_{t+1} = \alpha \beta z_t k_t^\alpha$ and
$c_t = (1 - \alpha\beta) z_t k_t^\alpha$, and in logs that is exactly linear (slide 5). The second half of
part 5 sets $\gamma = 2$ and $\delta = 0.1$, where no closed form exists.

**The system.** The function `F(x_next, x, par)` takes the variables of tomorrow and today, each as the vector
$\mathbf{x} = (\ln k, \ln z, \ln c)$ in logs with the **states first**, and returns the three residuals. Each
residual is "log of the left side minus log of the right side", so that the residuals are dimensionless and
a residual of $0.01$ means one percent. Along an equilibrium path without shocks every residual is zero.
The steady state is the vector $\bar{\mathbf{x}}$ with $\mathbf{F}(\bar{\mathbf{x}}, \bar{\mathbf{x}}) = \mathbf{0}$:
$\bar z = 1$, and $k^*$ and $c^*$ from problem set 1 (slide 8).

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    alpha: float = 0.3        # capital share
    beta: float = 0.96        # discount factor
    gamma: float = 1.0        # relative risk aversion, 1 is log utility
    delta: float = 1.0        # depreciation rate, 1 is full depreciation
    rho: float = 0.9          # persistence of log TFP
    sigma: float = 0.02       # standard deviation of the TFP innovation


par = EconomicParameters()    # γ = 1, δ = 1

In [ ]:
def steady_state(par):
    # The steady state (k*, 1, c*) in levels (slide 8, the formula of problem set 1)
    # Gap 1: k* from the Euler equation at k = k': alpha k^(alpha - 1) + 1 - delta = 1 / beta, solved for k
    k = ___
    c = k ** par.alpha - par.delta * k                                               # the resource constraint at k = k'
    return np.array([k, 1.0, c])

In [ ]:
def F(x_next, x, par):
    # The equilibrium conditions. x = (ln k, ln z, ln c) today, x_next the same tomorrow.
    # Returns the three residuals, each "log of the left side minus log of the right side" (slide 7)
    k, z, c = np.exp(x)                                                            # today, in levels
    k1, z1, c1 = np.exp(x_next)                                                    # tomorrow, in levels
    R1 = par.alpha * z1 * k1 ** (par.alpha - 1) + 1 - par.delta                    # gross return tomorrow
    return np.array([
        np.log(k1) - np.log(z * k ** par.alpha + (1 - par.delta) * k - c),         # resource constraint
        np.log(z1) - par.rho * np.log(z),                                           # TFP, without the shock
        # Gap 2: the Euler equation: log of u_c(c) = c^(-gamma) minus log of beta u_c(c') R'
        ___,
    ])

In [ ]:
xss = np.log(steady_state(par))                                                    # the steady state, in logs
print("steady state k*, z, c*:", steady_state(par))
print("F at the steady state:  ", F(xss, xss, par))

**Check (slide 8):** $k^* = 0.1689$ and $c^* = 0.4176$, and all three residuals are zero at the steady state
(up to $10^{-16}$). If the third residual is not zero, the Euler equation in gap 2 has a sign or a log wrong.

### 2. Linearization by finite differences (gaps 3 and 4)

*Slides 8 to 10.*

With $\hat{\mathbf{x}}_t = \ln \mathbf{x}_t - \ln \bar{\mathbf{x}}$ the first-order Taylor expansion of
$\mathbf{F}$ around the steady state is

$$
\mathbf{A} \, \mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \mathbf{B} \, \hat{\mathbf{x}}_t,
\qquad
\mathbf{A} = \frac{\partial \mathbf{F}}{\partial \hat{\mathbf{x}}_{t+1}} \Big|_{\text{ss}}, \quad
\mathbf{B} = - \frac{\partial \mathbf{F}}{\partial \hat{\mathbf{x}}_{t}} \Big|_{\text{ss}} .
$$

Since `F` takes logs as arguments, the derivatives with respect to the log deviations are the derivatives
with respect to its arguments. We compute them by **central differences**: column $j$ of a Jacobian is
$[\mathbf{f}(\mathbf{x}_0 + h \mathbf{e}_j) - \mathbf{f}(\mathbf{x}_0 - h \mathbf{e}_j)] / 2h$ with
$h = 10^{-6}$, which is accurate to about $10^{-10}$. The function `jacobian(f, x0)` works for any vector
function `f` of one vector argument. `linearize` applies it twice: to $\mathbf{F}$ as a function of tomorrow's
variables with today's fixed at the steady state, and the other way round.

The expectation is gone: it passes through the linear expansion, and $\mathbb{E}_t \varepsilon_{t+1} = 0$.
Nothing in $\mathbf{A}$ and $\mathbf{B}$ depends on $\sigma$ (certainty equivalence, slide 8).

In [ ]:
def jacobian(f, x0, h=1e-6):
    # The Jacobian of f at x0 by central differences: column j is df / dx_j (slide 8)
    n, m = len(x0), len(f(x0))
    J = np.zeros((m, n))
    for j in range(n):
        e = np.zeros(n)
        e[j] = h                                                                   # a step in direction j only
        # Gap 3: the central difference: f one step up minus f one step down, over the length of the two steps
        J[:, j] = ___
    return J

In [ ]:
def linearize(F, xss, par):
    # The matrices A and B of the linear system A E x' = B x, both evaluated at the steady state xss
    A = jacobian(lambda x_next: F(x_next, xss, par), xss)                          # derivative with respect to tomorrow
    # Gap 4: the derivative with respect to today, with the sign that moves it to the right-hand side (slide
    #        8)
    B = ___
    return A, B

In [ ]:
A, B = linearize(F, xss, par)
print("A =\n", A)
print("B =\n", B)
print("by hand: 1 / beta =", round(1 / par.beta, 4), " 1 / (alpha beta) =", round(1 / (par.alpha * par.beta), 4),
      " (1 - alpha beta) / (alpha beta) =", round((1 - par.alpha * par.beta) / (par.alpha * par.beta), 4))

**Check (slide 10 and the backup slide with the numbers):** the third row of $\mathbf{A}$ is $(0.7, -1, 1)$, the first row of $\mathbf{B}$ is
$(1.042, 3.472, -2.472)$, the second row of $\mathbf{B}$ is $(0, 0.9, 0)$, and the third $(0, 0, 1)$. These are
the hand-derived entries, reproduced to $10^{-10}$. A first row of $\mathbf{B}$ with the opposite
sign means gap 4 lacks the minus.

### 3. Blanchard and Kahn: eigenvalues, the count, $\mathbf{G}$ and $\mathbf{H}$ (gaps 5 and 6)

*Slides 14 and 15.*

Here $\mathbf{A}$ is invertible, so the system can be written as
$\mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \boldsymbol{\Gamma} \hat{\mathbf{x}}_t$ with
$\boldsymbol{\Gamma} = \mathbf{A}^{-1} \mathbf{B}$. The eigen-decomposition
$\boldsymbol{\Gamma} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^{-1}$ gives one eigenvalue per variable, each
a direction in which the system grows as $\lambda_i^t$. The solution is the one bounded path, and the
**counting rule** says when there is exactly one: as many stable eigenvalues ($|\lambda_i| < 1$) as states.

The recipe of slide 15, with the eigenvalues ordered so that the stable ones come first:
$\mathbf{W} = \mathbf{V}^{-1}$, its last $n_c$ rows $\mathbf{W}_u$ belong to the unstable eigenvalues, and
boundedness requires $\mathbf{W}_u \hat{\mathbf{x}}_t = \mathbf{0}$ in every period. Split $\mathbf{W}_u$ into
its state columns $\mathbf{W}_{us}$ and its control columns $\mathbf{W}_{uc}$, then

$$
\hat{c}_t = - \mathbf{W}_{uc}^{-1} \mathbf{W}_{us} \, \hat{\mathbf{s}}_t = \mathbf{G} \hat{\mathbf{s}}_t,
\qquad
\mathbf{H} = \boldsymbol{\Gamma}_{ss} + \boldsymbol{\Gamma}_{sc} \mathbf{G} .
$$

The function takes $\mathbf{A}$, $\mathbf{B}$, and the number of states, and returns $\mathbf{G}$,
$\mathbf{H}$, and the eigenvalues. Eigenvalues and eigenvectors can come out complex (they come in conjugate
pairs when they do), the result is real up to rounding, so the function returns the real parts.

In [ ]:
def solve_bk(A, B, n_states):
    # Blanchard and Kahn (slide 15): needs A invertible. Returns G (controls on states), H (law of motion of the
    # states), and the eigenvalues
    Gamma = solve(A, B)                                                            # E x' = Gamma x
    lam, V = eig(Gamma)                                                            # eigenvalues, eigenvectors as columns
    order = np.argsort(np.abs(lam))                                                # stable eigenvalues first
    lam, V = lam[order], V[:, order]
    # Gap 5: the number of stable eigenvalues: how many have modulus below one
    n_stable = ___
    if n_stable != n_states:
        print(f"Blanchard and Kahn: {n_stable} stable eigenvalues for {n_states} states")
    W = inv(V)                                                                     # rows are the left eigenvectors
    Wu = W[n_states:, :]                                                           # the unstable rows must annihilate x
    # Gap 6: the controls on the states, from W_us s + W_uc c = 0 (slide 15, step 2); the state columns are
    #        the first n_states
    G = ___
    H = Gamma[:n_states, :n_states] + Gamma[:n_states, n_states:] @ G              # the first n_states rows of Gamma
    return G.real, H.real, lam

In [ ]:
n_states = 2
G, H, lam = solve_bk(A, B, n_states)
print("eigenvalues:", lam)
print("G =", G)
print("H =\n", H)
print("largest difference to the closed form (alpha, 1):", np.max(np.abs(G - np.array([[par.alpha, 1.0]]))))

**Check (slides 14 and 15):** eigenvalues $0.3$, $0.9$, $3.472$, two stable for two states. $\mathbf{G} = (0.3, \, 1)$
and $\mathbf{H} = \bigl( \begin{smallmatrix} 0.3 & 1 \\ 0 & 0.9 \end{smallmatrix} \bigr)$: the closed form
$\hat{c}_t = \alpha \hat{k}_t + \hat{z}_t$ and $\hat{k}_{t+1} = \alpha \hat{k}_t + \hat{z}_t$ of slide 5, to
about $10^{-11}$, the precision of the finite differences. The three eigenvalues have names: $0.3 = \alpha$ is
how fast capital forgets its past, $0.9 = \rho$ is TFP, $3.472 = 1 / \alpha\beta$ is the unstable direction
that the choice of $\hat{c}_0$ switches off.

### 4. Klein: the generalized Schur decomposition (gaps 7 and 8)

*Slides 17 to 21.*

The generalized Schur (QZ) decomposition of the pair $(\mathbf{A}, \mathbf{B})$ writes
$\mathbf{Q}^{\mathsf{G}} \mathbf{A} \mathbf{Z} = \mathbf{S}$ and $\mathbf{Q}^{\mathsf{G}} \mathbf{B} \mathbf{Z} = \mathbf{T}$
with $\mathbf{S}$ and $\mathbf{T}$ upper triangular and $\mathbf{Q}$, $\mathbf{Z}$ unitary. The generalized
eigenvalues are $\lambda_i = t_{ii} / s_{ii}$, and a zero $s_{ii}$ is an infinite eigenvalue: a static equation,
which is eliminated like any other unstable direction. With the stable eigenvalues ordered first, slide 20 gives

$$
\mathbf{G} = \mathbf{Z}_{21} \mathbf{Z}_{11}^{-1}, \qquad
\mathbf{H} = \mathbf{Z}_{11} \mathbf{S}_{11}^{-1} \mathbf{T}_{11} \mathbf{Z}_{11}^{-1} ,
$$

where the index $1$ refers to the first $n_s$ rows or columns. Nothing in it inverts $\mathbf{A}$.

**One trap in the library call.** Both SciPy and Julia report the eigenvalues of the pair as
$\alpha_i / \beta_i$ with $\alpha_i = s_{ii}$ and $\beta_i = t_{ii}$, the reciprocal of our $\lambda_i$. Stable
in our sense, $|t_{ii}| < |s_{ii}|$, is therefore $|\beta_i| < |\alpha_i|$ in theirs, and that is the test to
hand to the sorting routine. The ready-made option "inside the unit circle" would select the wrong block.
The decomposition is done in complex arithmetic: complex eigenvalues come in pairs that the real version
keeps in $2 \times 2$ blocks, which would break the triangular reading. The result is real up to $10^{-16}$.

In [ ]:
def solve_klein(A, B, n_states):
    # Klein (2000), slide 20: the generalized Schur decomposition, A need not be invertible. Returns G, H, and the
    # eigenvalues (infinite ones appear as very large numbers)
    # Gap 7: the test for a stable eigenvalue, |t_ii| < |s_ii|, in the library's names: beta is t_ii, alpha is
    #        s_ii
    S, T, alpha, beta, Q, Z = ordqz(A, B, sort=lambda a, b: ___, output="complex")
    n_stable = np.sum(np.abs(beta) < np.abs(alpha))
    if n_stable != n_states:
        print(f"Klein: {n_stable} stable eigenvalues for {n_states} states")
    Z11, Z21 = Z[:n_states, :n_states], Z[n_states:, :n_states]                   # the first n_states columns of Z, split
    S11, T11 = S[:n_states, :n_states], T[:n_states, :n_states]                   # the stable block
    # Gap 8: the controls on the states (slide 20, step 2)
    G = ___
    H = Z11 @ solve(S11, T11) @ inv(Z11)                                           # the law of motion (slide 20, step 3)
    with np.errstate(divide="ignore", invalid="ignore"):
        eigs = beta / alpha                                                        # our lambda_i = t_ii / s_ii
    return G.real, H.real, eigs

In [ ]:
Gk, Hk, eigs = solve_klein(A, B, n_states)
print("eigenvalues:", eigs)
print("G =", Gk)
print("H =\n", Hk)
print("largest difference to Blanchard and Kahn:", np.max(np.abs(Gk - G)), np.max(np.abs(Hk - H)))

**Check (slide 20):** the same eigenvalues (in the order the decomposition chose), the same $\mathbf{G}$ and
$\mathbf{H}$ as in part 3, to $10^{-15}$.

**A model that only Klein can handle.** Write the same economy the way it is written in practice (slide 17),
with output, investment, and the return as named variables:

$$
y_t = z_t k_t^\alpha, \qquad i_t = y_t - c_t, \qquad k_{t+1} = (1 - \delta) k_t + i_t, \qquad
R_{t+1} = \alpha \, y_{t+1} / k_{t+1} + 1 - \delta .
$$

Now $\mathbf{x}_t = (k_t, z_t, c_t, y_t, i_t)$, still two states, and two of the five equations contain no
variable dated $t + 1$. Those are two zero rows in $\mathbf{A}$. Nothing to fill in here: `F_ext` is `F` with
the two static equations added, `linearize` and `solve_klein` are the functions from above.

In [ ]:
def F_ext(x_next, x, par):
    # The same model with output y and investment i as named variables, x = (ln k, ln z, ln c, ln y, ln i)
    k, z, c, y, i = np.exp(x)
    k1, z1, c1, y1, i1 = np.exp(x_next)
    R1 = par.alpha * y1 / k1 + 1 - par.delta                                       # the return, from output and capital
    return np.array([
        np.log(k1) - np.log((1 - par.delta) * k + i),                              # capital accumulation
        np.log(z1) - par.rho * np.log(z),                                           # TFP
        -par.gamma * np.log(c) - np.log(par.beta * c1 ** (-par.gamma) * R1),        # Euler equation
        np.log(y) - np.log(z) - par.alpha * np.log(k),                              # production: static
        np.log(i) - np.log(y - c),                                                  # investment: static
    ])


def steady_state_ext(par):
    k, z, c = steady_state(par)
    y = k ** par.alpha
    return np.array([k, z, c, y, y - c])

In [ ]:
xss_ext = np.log(steady_state_ext(par))
A_ext, B_ext = linearize(F_ext, xss_ext, par)
print("A_ext =\n", A_ext)
print("rank of A_ext:", np.linalg.matrix_rank(A_ext), "of", A_ext.shape[0])
G_ext, H_ext, eigs_ext = solve_klein(A_ext, B_ext, n_states)
print("eigenvalues:", eigs_ext)
print("G_ext (rows: c, y, i on k, z) =\n", G_ext)
print("H_ext =\n", H_ext)

**Check (slide 18 and the backup slide on the extended system):** $\mathbf{A}$ has rank $3$ of $5$, so
$\mathbf{A}^{-1} \mathbf{B}$ does not exist and `solve_bk` would stop with a singular matrix. The eigenvalues are
$0.3$, $0.9$, $3.472$, and two of the order $10^{17}$, which is how the library writes $\infty$. Two stable
ones for two states, the count still fits. $\mathbf{G}$ is now $3 \times 2$ (consumption, output, and
investment on the two states), and all three rows read $(0.3, \, 1)$: with $\gamma = 1$, $\delta = 1$, $c$, $y$, and $i$ are
all proportional to $z k^\alpha$. $\mathbf{H}$ is the one from before.

### 5. Using the solution: impulse responses, simulation, and a second economy (gaps 9 and 10)

*Slides 22 to 25.*

With $\mathbf{G}$ and $\mathbf{H}$ in hand everything else is a recursion. An **impulse response** starts the
states at a shock, $\hat{\mathbf{s}}_0 = (0, \, 0.01)$ for a one percent TFP shock, and iterates
$\hat{\mathbf{s}}_{t+1} = \mathbf{H} \hat{\mathbf{s}}_t$ with no further shocks, the controls following as
$\hat{c}_t = \mathbf{G} \hat{\mathbf{s}}_t$. A **simulation** draws $\varepsilon_t \sim N(0, \sigma^2)$ from a
seeded generator and adds $\varepsilon_{t+1}$ to log TFP in every step. Both functions take $\mathbf{G}$,
$\mathbf{H}$, and the number of states, so that they work for the extended model and for the second economy alike.

In [ ]:
def impulse_response(G, H, n_states, T, shock):
    # The paths of states and controls for T periods after the shock vector hits the states in period 0 (slide 22)
    s = np.zeros((T, n_states))                                                    # row t is the state vector in period t
    s[0] = shock
    for t in range(T - 1):
        # Gap 9: the law of motion: tomorrow's states from today's
        s[t + 1] = ___
    c = s @ G.T                                                                    # the controls in every period, row t is G s_t
    return s, c

In [ ]:
T = 25
s_irf, c_irf = impulse_response(G, H, n_states, T, np.array([0.0, 0.01]))
print("k_hat in percent, periods 0 to 5:", 100 * s_irf[:6, 0])
print("c_hat in percent, periods 0 to 5:", 100 * c_irf[:6, 0])

In [ ]:
# The exact path with gamma = 1, delta = 1 in levels, k' = alpha beta z k^alpha, after the same shock (slide 22)
kss, _, css = steady_state(par)
z_exact = np.exp(par.rho ** np.arange(T) * 0.01)                                   # ln z_t = rho^t 0.01, no further shocks
k_exact = np.zeros(T)
k_exact[0] = kss
for t in range(T - 1):
    k_exact[t + 1] = par.alpha * par.beta * z_exact[t] * k_exact[t] ** par.alpha
c_exact = (1 - par.alpha * par.beta) * z_exact * k_exact ** par.alpha
print("largest difference between the exact path and the linear solution, k:", np.max(np.abs(np.log(k_exact / kss) - s_irf[:, 0])))

In [ ]:
tt = np.arange(T)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(tt, 100 * s_irf[:, 1], color="gray", lw=1.5, label="TFP $\\hat{z}_t$")
ax.plot(tt, 100 * s_irf[:, 0], lw=1.8, label="capital $\\hat{k}_t$, linear solution")
ax.plot(tt, 100 * c_irf[:, 0], lw=1.8, label="consumption $\\hat{c}_t$, linear solution")
ax.plot(tt, 100 * np.log(k_exact / kss), "o", ms=4, mfc="white", color="C0", label="exact path")
ax.plot(tt, 100 * np.log(c_exact / css), "s", ms=4, mfc="white", color="C1")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("periods after the shock"); ax.set_ylabel("percent deviation from steady state")
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

**Check (slide 22):** consumption jumps by one percent on impact, capital reaches $1.2$ percent after two
periods ($0$, $1.0$, $1.2$, $1.17$, $1.08$, $0.98$ percent), and both decay at $\rho$. The exact path and the
linear solution differ by about $10^{-11}$: with $\gamma = 1$ and $\delta = 1$ the linear solution is the closed form.

**Simulation.** Now with shocks in every period. The seed makes the draw reproducible, the same seed gives the
same path on every machine. The exact economy in levels is fed with the same innovations.

In [ ]:
def simulate(G, H, n_states, T, sigma, seed):
    # A simulated path of T periods from the steady state, with innovations eps ~ N(0, sigma^2) to log TFP (slide 23)
    rng = np.random.default_rng(seed)                                              # seeded, so that the run is reproducible
    eps = sigma * rng.standard_normal(T)
    s = np.zeros((T, n_states))
    for t in range(T - 1):
        s[t + 1] = H @ s[t]
        # Gap 10: the innovation hits log TFP, the second state, in period t + 1
        ___
    return s, s @ G.T, eps

In [ ]:
def simulate_exact_bm(par, eps, kss):
    # The economy with gamma = 1, delta = 1 in levels, k' = alpha beta z k^alpha and c = (1 - alpha beta) z k^alpha, with the innovations eps
    T = len(eps)
    k, z, c = np.zeros(T), np.ones(T), np.zeros(T)
    k[0] = kss
    for t in range(T):
        c[t] = (1 - par.alpha * par.beta) * z[t] * k[t] ** par.alpha
        if t < T - 1:
            k[t + 1] = par.alpha * par.beta * z[t] * k[t] ** par.alpha
            z[t + 1] = np.exp(par.rho * np.log(z[t]) + eps[t + 1])
    return k, z, c

In [ ]:
T_sim = 200
s_sim, c_sim, eps = simulate(G, H, n_states, T_sim, par.sigma, 11)
k_ex, z_ex, c_ex = simulate_exact_bm(par, eps, kss)
print("standard deviations in percent, k_hat, c_hat, z_hat:", 100 * s_sim[:, 0].std(), 100 * c_sim[:, 0].std(), 100 * s_sim[:, 1].std())
print("largest difference between the exact path and the linear solution:", np.max(np.abs(np.log(k_ex / kss) - s_sim[:, 0])))

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 3.6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
axs[0].plot(100 * s_sim[:, 0], lw=1.4, label="$\\hat{k}_t$, linear solution")
axs[0].plot(100 * np.log(k_ex / kss), ".", ms=3, label="exact path, same shocks")
axs[0].plot(100 * s_sim[:, 1], color="gray", lw=0.8, label="$\\hat{z}_t$")
axs[0].axhline(0, color="k", lw=0.5); axs[0].set_ylabel("percent"); axs[0].legend(fontsize=8, ncol=3)
axs[1].plot(np.log(k_ex / kss) - s_sim[:, 0], color="k", lw=0.8)
axs[1].set_ylabel("exact minus linear", fontsize=8); axs[1].set_xlabel("period")
plt.tight_layout(); plt.show()

**Check (slide 23):** with seed $11$ the standard deviations are about $5.1$ percent for capital, $5.1$ for
consumption, and $3.7$ for TFP (another seed moves the first decimal). The exact path and the linear solution
differ by $10^{-11}$ over $200$ periods, with shocks of two percent: the replication is done.

**A second economy.** Two parameters change, $\gamma = 2$ and $\delta = 0.1$. The
code does not change at all. There is no closed form to compare with, so the yardstick is a **global
solution** of the deterministic version of the model ($\sigma = 0$), computed by the endogenous grid method of
week 5 on $400$ grid points. At first order the linear solution does not depend on $\sigma$ anyway, so the
deterministic policy is the right comparison for it. The solver is given. The Euler equation error $e(k)$ in
units of consumption is the one of problem set 1, now computed for the linear policy and for the global one.

In [ ]:
par2 = EconomicParameters(gamma=2.0, delta=0.1)                                   # the second economy
xss2 = np.log(steady_state(par2))
kss2, _, css2 = steady_state(par2)
A2, B2 = linearize(F, xss2, par2)
G2, H2, eigs2 = solve_klein(A2, B2, n_states)
print("steady state k*, z, c*:", steady_state(par2))
print("eigenvalues:", eigs2)
print("G =", G2)
print("H =\n", H2)

In [ ]:
T = 40
s1, c1 = impulse_response(G, H, n_states, T, np.array([0.0, 0.01]))               # gamma = 1, delta = 1
s2, c2 = impulse_response(G2, H2, n_states, T, np.array([0.0, 0.01]))             # gamma = 2, delta = 0.1
print("gamma = 2, delta = 0.1: c_hat on impact", 100 * c2[0, 0], "percent, k_hat peaks at", 100 * s2[:, 0].max(), "percent in period", s2[:, 0].argmax())
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
tt = np.arange(T)
for ax, (s_, c_), title in ((axs[0], (s1, c1), "$\\gamma = 1$, $\\delta = 1$"), (axs[1], (s2, c2), "$\\gamma = 2$, $\\delta = 0.1$")):
    ax.plot(tt, 100 * s_[:, 1], color="gray", lw=1.2, label="$\\hat{z}_t$")
    ax.plot(tt, 100 * s_[:, 0], lw=1.8, label="$\\hat{k}_t$")
    ax.plot(tt, 100 * c_[:, 0], lw=1.8, label="$\\hat{c}_t$")
    ax.axhline(0, color="k", lw=0.5); ax.set_title(title, fontsize=10); ax.set_xlabel("periods after the shock")
axs[0].set_ylabel("percent deviation"); axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
def interp_extrap(x, xp, fp):
    # np.interp, but linear extrapolation beyond the ends of the grid instead of a flat continuation
    y = np.interp(x, xp, fp)
    lo, hi = x < xp[0], x > xp[-1]
    y[lo] = fp[0] + (x[lo] - xp[0]) * (fp[1] - fp[0]) / (xp[1] - xp[0])
    y[hi] = fp[-1] + (x[hi] - xp[-1]) * (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return y


def solve_global(par, kgrid, tol=1e-10, maxit=10_000):
    # The consumption policy c(k) of the deterministic growth model on kgrid, by the endogenous grid method of week 5:
    # kgrid plays tomorrow's capital, the Euler equation gives today's consumption, the resource constraint today's capital
    m_grid = kgrid ** par.alpha + (1 - par.delta) * kgrid                          # resources as a function of k, increasing
    c = m_grid - kgrid                                                             # start: keep capital constant
    for it in range(maxit):
        R = par.alpha * kgrid ** (par.alpha - 1) + 1 - par.delta                  # the return when k' is a grid point
        c_today = (par.beta * R * c ** (-par.gamma)) ** (-1 / par.gamma)          # the Euler equation, c' = c(k') on the grid
        m_today = c_today + kgrid                                                  # the resources that make this k' optimal
        k_today = interp_extrap(m_today, m_grid, kgrid)                            # the k that has those resources
        c_new = interp_extrap(kgrid, k_today, c_today)                             # back onto the fixed grid
        dist = np.max(np.abs(c_new - c))
        c = c_new
        if dist < tol:
            break
    return c, it + 1


def euler_error(kgrid, c, par):
    # The Euler equation error of a consumption policy c on kgrid, in units of consumption (problem set 1), sigma = 0
    k_next = kgrid ** par.alpha + (1 - par.delta) * kgrid - c
    c_next = np.interp(k_next, kgrid, c)
    R = par.alpha * k_next ** (par.alpha - 1) + 1 - par.delta
    return 1 - (par.beta * R) ** (-1 / par.gamma) * c_next / c

In [ ]:
kgrid = np.linspace(0.1 * kss2, 2 * kss2, 400)                                     # the grid of problem set 1
c_global, it = solve_global(par2, kgrid)
c_linear = css2 * np.exp(G2[0, 0] * np.log(kgrid / kss2))                          # the linear policy in levels, at z = 1
e_linear = euler_error(kgrid, c_linear, par2)
e_global = euler_error(kgrid, c_global, par2)
print(f"global solution: {it} iterations")
print(" k / k*   c linear   c global   difference   |e| linear   |e| global")
for f in (0.1, 0.5, 0.9, 1.0, 1.1, 1.5, 2.0):
    i = np.argmin(np.abs(kgrid - f * kss2))
    print(f"{f:6.1f} {c_linear[i]:10.4f} {c_global[i]:10.4f} {100 * (c_linear[i] / c_global[i] - 1):+9.2f} %  {abs(e_linear[i]):10.1e} {abs(e_global[i]):11.1e}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(kgrid / kss2, c_global, color="k", lw=1.6, label="global solution")
axs[0].plot(kgrid / kss2, c_linear, lw=1.6, ls="--", label="linear solution")
axs[0].plot(1, css2, "o", color="C1", label="steady state")
axs[0].set_xlabel("$k / k^*$"); axs[0].set_ylabel("consumption $c(k)$"); axs[0].legend(fontsize=8)
axs[1].semilogy(kgrid / kss2, np.abs(e_linear) + 1e-16, lw=1.6, label="linear solution")
axs[1].semilogy(kgrid / kss2, np.abs(e_global) + 1e-16, color="k", lw=1.2, label="global solution")
axs[1].set_ylim(1e-8, 1); axs[1].set_xlabel("$k / k^*$"); axs[1].set_ylabel("Euler equation error $|e(k)|$"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Check (slides 24 and 25):** $k^* = 2.921$, $c^* = 1.087$, eigenvalues $0.893$, $0.9$, $1.166$,
$\mathbf{G} = (0.399, \, 0.562)$. Consumption rises by $0.56$ percent on impact and capital peaks at $0.99$ percent
in period $9$. The global solution takes $82$ iterations. At $k = 0.9 \, k^*$ and $1.1 \, k^*$ the linear policy
is $0.02$ percent off the global one and its Euler equation error is about $10^{-4}$, at $0.5 \, k^*$ and
$2 \, k^*$ it is one percent off with an error of a few $10^{-3}$, at $0.1 \, k^*$ ten percent. The global
solution has errors of $10^{-6}$ everywhere (the interpolation on $400$ points). Exact at the steady state,
and the error grows with the square of the distance: that is what a first-order solution is.

**Experiment: the persistence of TFP.** Solve the economy with $\gamma = 2$ and $\delta = 0.1$ for $\rho \in \{0.5, 0.9, 0.99\}$ and plot
the impulse responses of capital and consumption next to each other. Look at what moves with $\rho$ and what
does not: the coefficient of $\hat{z}_t$ in $\mathbf{G}$, the period in which capital peaks, and the
eigenvalues. Then change $\rho$ with $\gamma = 1$ and $\delta = 1$ as well and check what happens to $\mathbf{G}$ there.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for rho in (0.5, 0.9, 0.99):
    par_rho = EconomicParameters(gamma=2.0, delta=0.1, rho=rho)
    Ap, Bp = linearize(F, np.log(steady_state(par_rho)), par_rho)
    Gp, Hp, ep = solve_klein(Ap, Bp, n_states)
    sp, cp = impulse_response(Gp, Hp, n_states, 80, np.array([0.0, 0.01]))
    print(f"rho = {rho}: G = {Gp[0]}, eigenvalues {np.sort(np.abs(ep))}, capital peaks at {100 * sp[:, 0].max():.2f} percent in period {sp[:, 0].argmax()}")
    axs[0].plot(100 * sp[:, 0], label=f"$\\rho = {rho}$")
    axs[1].plot(100 * cp[:, 0], label=f"$\\rho = {rho}$")
axs[0].set_title("capital $\\hat{k}_t$", fontsize=10); axs[1].set_title("consumption $\\hat{c}_t$", fontsize=10)
for ax in axs:
    ax.set_xlabel("periods after the shock"); ax.axhline(0, color="k", lw=0.5)
axs[0].set_ylabel("percent deviation"); axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()